## 📊 Data Pipeline Overview

This pipeline is designed to **incrementally fetch, process, and store data** for forecasting models.  
It integrates multiple sources and ensures weekly updates to the main dataset.


### 🔎 Data Sources
- **GDELT GKG** → Global news knowledge graph for event and sentiment features.  
- **Google Trends** → Search interest data for demand and hype signals. 
-- **Holidays and Macro Economic Data** 


### ⚙️ Processing Steps
1. **Incremental Fetching**  
   - Retrieve new records weekly.  
   - Include an overlap window for normalization and consistency.  

2. **Feature Engineering**  
   - Calculate all required features for each forecasting model.  
   - Normalize and align datasets across countries and categories.  

3. **Cloud Storage**  
   - Store processed `.csv` and `.pkl` files in **Cloudflare R2**.  
   - Maintain versioned datasets for reproducibility.  

4. **Database Update**  
   - Insert the latest week’s data into the **main TiDB table**.  
   - Ensure historical consistency while keeping the pipeline lightweight.  

---

### ✅ Key Benefits
- Automated **weekly updates**.  
- Cloud‑native storage (no local dependency).  
- Scalable feature computation for multiple models.  
- Reliable incremental design with overlap handling.  


In [89]:
# importing libraries
import os
import sys
from datetime import datetime, timedelta,date
import time
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
import pymysql
from pytrends.request import TrendReq
import requests
from requests.adapters import HTTPAdapter
from urllib3.util import Retry
from pytrends.request import TrendReq
from google.cloud import bigquery

In [2]:
# ------------------------------------------------------------------
# 1. Load Environment Variables
# ------------------------------------------------------------------
from dotenv import load_dotenv
import os, sys
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime

load_dotenv()

DB_HOST = os.getenv("DB_HOST")
DB_PORT = int(os.getenv("DB_PORT", 4000))
DB_USERNAME = os.getenv("DB_USERNAME")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_DATABASE = os.getenv("DB_DATABASE")

# Verify all variables are loaded
print("\n=== Connection Details ===")
print(f"Host: {DB_HOST}")
print(f"Port: {DB_PORT}")
print(f"Username: {DB_USERNAME}")
print(f"Database: {DB_DATABASE}")

if not all([DB_HOST, DB_USERNAME, DB_PASSWORD, DB_DATABASE]):
    print("\n✗ ERROR: Some environment variables are missing!")
    print("Check your .env file has all required values")
    sys.exit(1)

print("\n✓ All variables loaded!")

# ------------------------------------------------------------------
# 2. Connect to TiDB Database & Get Last Date
# ------------------------------------------------------------------
# Create SQLAlchemy connection URL using PyMySQL
connection_url = (
    f"mysql+pymysql://{DB_USERNAME}:{DB_PASSWORD}@"
    f"{DB_HOST}:{DB_PORT}/{DB_DATABASE}"
)

engine = create_engine(
    connection_url,
    connect_args={"ssl": {}} if "tidbcloud" in DB_HOST else {}
)

# ✅ FIX: Escaped '%' as '%%' so SQLAlchemy won't throw format character error
query = text("SELECT MAX(STR_TO_DATE(week_start, '%%Y-%%m-%%d')) AS last_date FROM main;")

try:
    with engine.connect() as connection:
        last_date_df = pd.read_sql(query, connection)

    last_recorded_date = last_date_df["last_date"].iloc[0]

    if pd.isna(last_recorded_date):
        last_recorded_date = datetime(2026, 8, 28).date()
    elif isinstance(last_recorded_date, (pd.Timestamp, datetime)):
        last_recorded_date = last_recorded_date.date()

    print(f"✓ Last recorded date in TiDB: {last_recorded_date}")

except Exception as e:
    print(f"\n✗ Database query failed: {e}")
    sys.exit(1)


=== Connection Details ===
Host: gateway01.ap-northeast-1.prod.aws.tidbcloud.com
Port: 4000
Username: qgNsbp7EdsPhLkz.root
Database: GDELT&TRENDS

✓ All variables loaded!
✓ Last recorded date in TiDB: 2026-08-28


### Downloading Google Trends Data

In [3]:
# ------------------------------------------------------------------
# 1. Configuration & Constants
# ------------------------------------------------------------------
COUNTRY_MAP = {
    "US": "United_States",
    "IN": "India",
    "FR": "France",
    "NG": "Nigeria",
    "GB": "United_Kingdom",
    "CA": "Canada",
    "AU": "Australia",
    "BR": "Brazil",
    "MX": "Mexico",
    "ZA": "South_Africa",
    "KE": "Kenya",
    "SG": "Singapore",
    "AE": "United_Arab_Emirates",
    "CN": "China",
}

EXPANDED_CATEGORY_BATCHES = {
    "Nutrition_Diets": [
        ["diet", "nutrition", "vegan", "vegetarian", "plant based"],
        ["keto", "paleo", "low carb", "intermittent fasting"],
        ["detox", "superfood", "organic", "weight loss"],
        ["supplement", "protein powder", "whey", "creatine"],
        ["vitamin", "minerals", "probiotics", "functional food"],
    ],
    "Fitness_Wearables": [
        ["fitness", "exercise", "workout", "training", "gym"],
        ["yoga", "pilates", "aerobics", "hiit"],
        ["crossfit", "cardio", "running", "cycling"],
        ["wearable", "smartwatch", "fitness tracker", "garmin"],
        ["fitbit", "apple watch", "heart rate monitor", "step counter"],
    ],
    "Fashion_Beauty": [
        ["fashion", "clothing", "apparel", "style", "designer"],
        ["luxury fashion", "fast fashion", "streetwear", "athleisure"],
        ["skincare", "makeup", "cosmetics", "moisturizer"],
        ["anti aging", "haircare", "shampoo", "fragrance"],
        ["perfume", "beauty treatment", "sneakers", "jewelry"],
    ],
}


In [4]:
# ------------------------------------------------------------------
# 3. Compute Google Trends Extraction Date Window
# ------------------------------------------------------------------

# Keep the database's latest recorded week as the reference point
google_trends_last_recorded_date = last_recorded_date

# Today
today = datetime.now().date()

# Sunday = 6 in Python's weekday() convention
# Find the current Sunday-starting week
current_week_start = today - timedelta(
    days=(today.weekday() + 1) % 7
)

# Current week is incomplete.
# Therefore, latest completed week is the previous Sunday.
google_trends_end_date = current_week_start - timedelta(days=7)

# ------------------------------------------------------------------
# Google Trends needs historical overlap for normalization
# ------------------------------------------------------------------

NORMALIZATION_OVERLAP_WEEKS = 8

google_trends_start_date = (
    google_trends_last_recorded_date
    - timedelta(days=NORMALIZATION_OVERLAP_WEEKS * 7)
)

# ------------------------------------------------------------------
# Check extraction range
# ------------------------------------------------------------------

if google_trends_last_recorded_date >= google_trends_end_date:

    print(
        f"✅ Google Trends data is already up to date as of "
        f"{google_trends_last_recorded_date}. "
        f"No completed weeks available to fetch."
    )

    exit()

# ------------------------------------------------------------------
# Google Trends timeframe
# ------------------------------------------------------------------

timeframe_str = (
    f"{google_trends_start_date.strftime('%Y-%m-%d')} "
    f"{google_trends_end_date.strftime('%Y-%m-%d')}"
)

print(f"📅 Last recorded week       : {google_trends_last_recorded_date}")
print(f"📅 Current week start       : {current_week_start}")
print(f"📅 Last completed week      : {google_trends_end_date}")
print(f"🔗 Normalization overlap    : {NORMALIZATION_OVERLAP_WEEKS} weeks")
print(f"🚀 Google Trends fetch range: {timeframe_str}\n")

📅 Last recorded week       : 2026-08-28
📅 Current week start       : 2026-09-27
📅 Last completed week      : 2026-09-20
🔗 Normalization overlap    : 8 weeks
🚀 Google Trends fetch range: 2026-07-03 2026-09-20



In [5]:
# ------------------------------------------------------------
# Create PyTrends session
# ------------------------------------------------------------
def create_pytrends_session():
    session = requests.Session()

    retries = Retry(
        total=3,
        connect=3,
        read=3,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET", "POST"],
    )

    adapter = HTTPAdapter(max_retries=retries)

    session.mount("https://", adapter)
    session.mount("http://", adapter)

    return TrendReq(hl="en-US", tz=360, requests_args={"verify": True})


# ------------------------------------------------------------
# Fetch Google Trends with Sunday Weekly Resampling
# ------------------------------------------------------------

pytrends = create_pytrends_session()
all_new_records = []

for geo_code, country_name in COUNTRY_MAP.items():
    print(f"\n========== {country_name} ({geo_code}) ==========")

    for cat_name, batches in EXPANDED_CATEGORY_BATCHES.items():
        print(f"\nFetching category: {cat_name}")
        category_batch_dfs = []

        for batch_idx, keyword_batch in enumerate(batches):
            success = False

            for attempt in range(1, 4):
                try:
                    print(
                        f"   Batch {batch_idx + 1}/{len(batches)} "
                        f"| Attempt {attempt}"
                    )

                    pytrends.build_payload(
                        keyword_batch, geo=geo_code, timeframe=timeframe_str
                    )

                    df_raw = pytrends.interest_over_time()

                    if df_raw.empty:
                        print(
                            f"   ⚠️ Empty response for batch {batch_idx + 1}"
                        )
                        break

                    if "isPartial" in df_raw.columns:
                        df_raw = df_raw.drop(columns=["isPartial"])

                    batch_cols = [
                        kw for kw in keyword_batch if kw in df_raw.columns
                    ]

                    if not batch_cols:
                        print("   ⚠️ No keyword columns returned")
                        break

                    # --------------------------------------------------------
                    #  Resample daily data to weekly starting on Sunday
                    # --------------------------------------------------------
                    df_weekly = (
                        df_raw[batch_cols].resample("W-SUN").mean().round(2)
                    )

                    score_col = f"batch_{batch_idx}_score"
                    df_weekly[score_col] = df_weekly[batch_cols].mean(axis=1)

                    category_batch_dfs.append(df_weekly[[score_col]])

                    success = True
                    print("   ✓ Success")
                    break

                except Exception as e:
                    print(f"   ⚠️ Attempt {attempt} failed: {e}")
                    if attempt < 3:
                        wait_time = 5 * attempt
                        print(f"   Waiting {wait_time}s...")
                        time.sleep(wait_time)

            # Small delay between requests to avoid rate limits
            time.sleep(2)

        # ----------------------------------------------------
        # Combine batches for category
        # ----------------------------------------------------
        if category_batch_dfs:
            df_cat_merged = pd.concat(category_batch_dfs, axis=1)

            df_cat_merged["search_interest"] = df_cat_merged.mean(axis=1)

            max_val = df_cat_merged["search_interest"].max()

            if max_val > 0:
                df_cat_merged["search_interest"] = (
                    df_cat_merged["search_interest"] / max_val * 100
                ).round(2)

            df_cat_merged = df_cat_merged.reset_index().rename(
                columns={"date": "week_start"}
            )

            # Ensure date column is formatted as standard string (YYYY-MM-DD)
            df_cat_merged["week_start"] = pd.to_datetime(
                df_cat_merged["week_start"]
            ).dt.strftime("%Y-%m-%d")

            df_cat_merged["country_code"] = geo_code
            df_cat_merged["country_name"] = country_name
            df_cat_merged["category"] = cat_name

            all_new_records.append(
                df_cat_merged[
                    [
                        "week_start",
                        "country_code",
                        "country_name",
                        "category",
                        "search_interest",
                    ]
                ]
            )

            print(f"   ✓ Category completed: {cat_name}")
        else:
            print(f"   ❌ No data collected: {cat_name}")


========== United_States (US) ==========

Fetching category: Nutrition_Diets
   Batch 1/5 | Attempt 1
   ✓ Success
   Batch 2/5 | Attempt 1
   ✓ Success
   Batch 3/5 | Attempt 1
   ✓ Success
   Batch 4/5 | Attempt 1
   ✓ Success
   Batch 5/5 | Attempt 1
   ✓ Success
   ✓ Category completed: Nutrition_Diets

Fetching category: Fitness_Wearables
   Batch 1/5 | Attempt 1
   ✓ Success
   Batch 2/5 | Attempt 1
   ✓ Success
   Batch 3/5 | Attempt 1
   ✓ Success
   Batch 4/5 | Attempt 1
   ✓ Success
   Batch 5/5 | Attempt 1
   ✓ Success
   ✓ Category completed: Fitness_Wearables

Fetching category: Fashion_Beauty
   Batch 1/5 | Attempt 1
   ✓ Success
   Batch 2/5 | Attempt 1
   ✓ Success
   Batch 3/5 | Attempt 1
   ✓ Success
   Batch 4/5 | Attempt 1
   ✓ Success
   Batch 5/5 | Attempt 1
   ✓ Success
   ✓ Category completed: Fashion_Beauty

========== India (IN) ==========

Fetching category: Nutrition_Diets
   Batch 1/5 | Attempt 1
   ✓ Success
   Batch 2/5 | Attempt 1
   ✓ Success
   Batch 

In [6]:
# ------------------------------------------------------------
# 4. Consolidate Final Output
# ------------------------------------------------------------
if all_new_records:
    df_final = pd.concat(all_new_records, ignore_index=True)
    
    # Save directly to CSV for the normalization pipeline
    df_final.to_csv("New_fetched_data.csv", index=False)
    
    print("\n" + "=" * 65)
    print(f"✅ DYNAMIC FETCH COMPLETE: {len(df_final):,} total records saved")
    print("📁 Exported to : New_fetched_data.csv")
    print("=" * 65)
else:
    df_final = pd.DataFrame(
        columns=["week_start", "country_code", "country_name", "category", "search_interest"]
    )
    df_final.to_csv("New_fetched_data.csv", index=False)
    print("\nℹ️ No records fetched. Created empty New_fetched_data.csv")


✅ DYNAMIC FETCH COMPLETE: 504 total records saved
📁 Exported to : New_fetched_data.csv


In [7]:
# ------------------------------------------------------------------
# 5. Output Preview
# ------------------------------------------------------------------
if all_new_records:
    df_raw_incremental = pd.concat(all_new_records, ignore_index=True)
    df_raw_incremental["week_start"] = pd.to_datetime(df_raw_incremental["week_start"]).dt.strftime("%Y-%m-%d")
    print("\n" + "="*65)
    print(f"✅ INCREMENTAL DATA FETCHED: {len(df_raw_incremental):,} records")
    print(df_raw_incremental.head(10))
    print("="*65)


✅ INCREMENTAL DATA FETCHED: 504 records
   week_start country_code   country_name         category  search_interest
0  2026-07-05           US  United_States  Nutrition_Diets            97.90
1  2026-07-12           US  United_States  Nutrition_Diets           100.00
2  2026-07-19           US  United_States  Nutrition_Diets            92.72
3  2026-07-26           US  United_States  Nutrition_Diets            82.93
4  2026-08-02           US  United_States  Nutrition_Diets            79.17
5  2026-08-09           US  United_States  Nutrition_Diets            78.72
6  2026-08-16           US  United_States  Nutrition_Diets            77.62
7  2026-08-23           US  United_States  Nutrition_Diets            75.66
8  2026-08-30           US  United_States  Nutrition_Diets            77.24
9  2026-09-06           US  United_States  Nutrition_Diets            77.85


In [37]:
# saving data to file
df_raw_incremental.to_csv("New_fetched_data.csv",index=False)

In [38]:
df_raw_incremental = pd.read_csv('New_fetched_data.csv')

In [39]:
df_raw_incremental['week_start'].min()

'2026-07-05'

In [40]:
df_raw_incremental['week_start'].max()

'2026-09-20'

### Normalization

In [41]:
import os
import sys
import pandas as pd
import numpy as np
from datetime import datetime
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# ------------------------------------------------------------------
# 1. Environment & Database Connection Setup
# ------------------------------------------------------------------
load_dotenv()

DB_HOST = os.getenv("DB_HOST")
DB_PORT = int(os.getenv("DB_PORT", 4000))
DB_USERNAME = os.getenv("DB_USERNAME")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_DATABASE = os.getenv("DB_DATABASE")

if not all([DB_HOST, DB_USERNAME, DB_PASSWORD, DB_DATABASE]):
    print("\n✗ ERROR: Missing TiDB environment variables in .env!")
    sys.exit(1)

connection_url = (
    f"mysql+pymysql://{DB_USERNAME}:{DB_PASSWORD}@"
    f"{DB_HOST}:{DB_PORT}/{DB_DATABASE}"
)

engine = create_engine(
    connection_url,
    connect_args={"ssl": {}} if "tidbcloud" in DB_HOST else {}
)


# ------------------------------------------------------------------
# 2. Load Historical Baseline Data (`main` Table)
# ------------------------------------------------------------------
query = text("""
SELECT
    week_start,
    country_name,
    category,
    search_interest AS Baseline_Interest
FROM main;
""")

try:
    with engine.connect() as connection:
        df_baseline = pd.read_sql(query, connection)
    print(f"✓ Baseline data successfully loaded from TiDB. Rows: {len(df_baseline):,}")
except Exception as e:
    print(f"\n✗ Failed to load baseline data from TiDB: {e}")
    sys.exit(1)

# Clean baseline data
df_baseline.columns = df_baseline.columns.str.lower()
df_baseline = df_baseline.rename(columns={"baseline_interest": "Baseline_Interest"})

df_baseline["week_start"] = pd.to_datetime(df_baseline["week_start"], errors="coerce")
df_baseline["Baseline_Interest"] = pd.to_numeric(df_baseline["Baseline_Interest"], errors="coerce")

df_baseline = df_baseline.dropna(
    subset=["week_start", "country_name", "category", "Baseline_Interest"]
).drop_duplicates(
    subset=["week_start", "country_name", "category"],
    keep="last"
)


# ------------------------------------------------------------------
# 3. Load Newly Fetched Data (`New_fetched_data.csv`)
# ------------------------------------------------------------------
try:
    df_incremental = pd.read_csv("New_fetched_data.csv")
    print(f"✓ Incremental CSV loaded. Rows: {len(df_incremental):,}")
except Exception as e:
    print(f"\n✗ Failed to load 'New_fetched_data.csv': {e}")
    sys.exit(1)

# Clean incremental data
df_incremental.columns = df_incremental.columns.str.lower()

df_incremental["week_start"] = pd.to_datetime(df_incremental["week_start"], errors="coerce")
df_incremental["search_interest"] = pd.to_numeric(df_incremental["search_interest"], errors="coerce")

df_incremental = df_incremental.dropna(
    subset=["week_start", "country_name", "category", "search_interest"]
)
df_incremental["search_interest"] = df_incremental["search_interest"].clip(lower=0)

df_incremental = (
    df_incremental
    .sort_values("week_start")
    .drop_duplicates(
        subset=["week_start", "country_name", "category"],
        keep="last"
    )
)


# ------------------------------------------------------------------
# 4. Identify Historical Boundary & Check Horizons
# ------------------------------------------------------------------
historical_end_date = df_baseline["week_start"].max()

print("\n" + "=" * 70)
print("DATABASE & INCREMENTAL HORIZON CHECK")
print("=" * 70)
print(f"Latest historical week in DB : {historical_end_date.date()}")
print(f"Google Trends first week     : {df_incremental['week_start'].min().date()}")
print(f"Google Trends last week      : {df_incremental['week_start'].max().date()}")
print("=" * 70)


# ------------------------------------------------------------------
# 5. Identify Overlap Weeks & Calculate Weekly Ratios
# ------------------------------------------------------------------
df_overlap = pd.merge(
    df_baseline[["week_start", "country_name", "category", "Baseline_Interest"]],
    df_incremental[["week_start", "country_name", "category", "search_interest"]],
    on=["week_start", "country_name", "category"],
    how="inner"
).rename(columns={"search_interest": "new_interest"})

# Keep up to recent N overlap weeks for ratio computation
OVERLAP_WEEKS = 8
df_overlap = df_overlap.sort_values(["country_name", "category", "week_start"])
df_overlap = (
    df_overlap
    .groupby(["country_name", "category"], group_keys=False)
    .tail(OVERLAP_WEEKS)
)

valid_overlap = df_overlap[
    (df_overlap["Baseline_Interest"] > 0) &
    (df_overlap["new_interest"] > 0)
].copy()

valid_overlap["Weekly_Ratio"] = (
    valid_overlap["Baseline_Interest"] / valid_overlap["new_interest"]
)


# ------------------------------------------------------------------
# 6. Calculate & Validate Median Ratios (With Fallback)
# ------------------------------------------------------------------
MIN_OVERLAP_WEEKS = 3

# Primary Category Ratio
df_ratios = (
    valid_overlap
    .groupby(["country_name", "category"], as_index=False)
    .agg(
        Ratio=("Weekly_Ratio", "median"),
        Overlap_Count=("Weekly_Ratio", "count")
    )
)
df_ratios_valid = df_ratios[df_ratios["Overlap_Count"] >= MIN_OVERLAP_WEEKS].copy()

# Country-Level Fallback Ratio
df_country_ratios = (
    valid_overlap
    .groupby("country_name", as_index=False)
    .agg(Country_Ratio=("Weekly_Ratio", "median"))
)


# ------------------------------------------------------------------
# 7. Attach Ratios & Normalize Incremental Data
# ------------------------------------------------------------------
df_scaled = pd.merge(
    df_incremental,
    df_ratios_valid[["country_name", "category", "Ratio"]],
    on=["country_name", "category"],
    how="left"
)

# Apply Fallbacks: Category -> Country -> 1.0 Default
df_scaled = pd.merge(df_scaled, df_country_ratios, on="country_name", how="left")
df_scaled["Ratio"] = df_scaled["Ratio"].fillna(df_scaled["Country_Ratio"]).fillna(1.0)
df_scaled = df_scaled.drop(columns=["Country_Ratio"])

df_scaled["Normalized_Search_Interest"] = (
    (df_scaled["search_interest"] * df_scaled["Ratio"]).round(2)
)


# ------------------------------------------------------------------
# 8. Combine Continuous Series
# ------------------------------------------------------------------
df_baseline_full = df_baseline.rename(columns={"Baseline_Interest": "search_interest"}).copy()

df_incremental_normalized = (
    df_scaled.drop(columns=["search_interest", "Ratio"])
    .rename(columns={"Normalized_Search_Interest": "search_interest"})
)

df_full = pd.concat([df_baseline_full, df_incremental_normalized], ignore_index=True)

df_full["week_start"] = pd.to_datetime(df_full["week_start"])

# Retain normalized updates for boundary overlap weeks while keeping full continuous history
df_full = (
    df_full
    .sort_values("week_start")
    .drop_duplicates(subset=["country_name", "category", "week_start"], keep="last")
    .sort_values(by=["country_name", "category", "week_start"])
    .reset_index(drop=True)
)


# ------------------------------------------------------------------
# 9. Feature Engineering Across Continuous Series
# ------------------------------------------------------------------
grouped = df_full.groupby(["country_name", "category"])

# Search Velocity and Acceleration
df_full["search_velocity"] = grouped["search_interest"].diff()
df_full["search_acceleration"] = grouped["search_velocity"].diff()

# Lags (1, 2, 4 weeks)
for lag in [1, 2, 4]:
    df_full[f"search_interest_lag_{lag}"] = grouped["search_interest"].shift(lag)

df_full['search_velocity_lag'] = grouped["search_velocity"].shift(1)

df_full["search_velocity_lag1"] = grouped["search_velocity"].shift(1)
df_full["search_velocity_lag2"] = grouped["search_velocity"].shift(2)
df_full["search_acceleration_lag"] = grouped["search_acceleration"].shift(1)

# Rolling 4-week Statistics
df_full["search_interest_roll_mean_4w"] = grouped["search_interest"].transform(
    lambda x: x.shift(1).rolling(4).mean()
)
df_full["search_interest_roll_std_4w"] = grouped["search_interest"].transform(
    lambda x: x.shift(1).rolling(4).std()
)


# ------------------------------------------------------------------
# 10. Trim to Target Horizon & Export Features
# ------------------------------------------------------------------
# Filter for strictly NEW weeks (> historical_end_date)
# Historical rows are used strictly for rolling windows and lags, then discarded here.
df_final_incremental = df_full[df_full["week_start"] > historical_end_date].copy()

# Format date column for export
df_final_incremental["week_start"] = df_final_incremental["week_start"].dt.strftime("%Y-%m-%d")

df_final_incremental.to_csv("Normalized_incremental_data_with_features.csv", index=False)

print("\n" + "=" * 70)
print("✅ NORMALIZATION & FEATURE ENGINEERING COMPLETE")
print(f"📊 Exported Rows      : {len(df_final_incremental):,}")
if not df_final_incremental.empty:
    print(f"📅 Weeks Normalized    : {df_final_incremental['week_start'].min()} to {df_final_incremental['week_start'].max()}")
print("📁 Saved Output File  : Normalized_incremental_data_with_features.csv")
print("=" * 70)

✓ Baseline data successfully loaded from TiDB. Rows: 8,022
✓ Incremental CSV loaded. Rows: 504

DATABASE & INCREMENTAL HORIZON CHECK
Latest historical week in DB : 2026-08-23
Google Trends first week     : 2026-07-05
Google Trends last week      : 2026-09-20

✅ NORMALIZATION & FEATURE ENGINEERING COMPLETE
📊 Exported Rows      : 168
📅 Weeks Normalized    : 2026-08-30 to 2026-09-20
📁 Saved Output File  : Normalized_incremental_data_with_features.csv


### GDELT GKG data

In [42]:
# ------------------------------------------------------------------
#  Compute Extraction Date Window
# ------------------------------------------------------------------

# Include the last recorded week again
fetch_start_date = last_recorded_date

# Today
today = datetime.now().date()

# Sunday = 6 in Python's weekday() convention
# Find the most recent COMPLETED Sunday-starting week.
# current week started Sunday 2026-09-20
# therefore the latest completed week started Sunday 2026-09-13

current_week_start = today - timedelta(
    days=(today.weekday() + 1) % 7
)

# The current week is incomplete, so use the previous Sunday
fetch_end_date = current_week_start - timedelta(days=7)


# ------------------------------------------------------------------
# Check extraction range
# ------------------------------------------------------------------

if fetch_start_date > fetch_end_date:

    print(
        f"✅ Data is already up to date as of "
        f"{last_recorded_date}. "
        f"No completed weeks available to fetch."
    )

    exit()


# ------------------------------------------------------------------
# Google Trends timeframe
# ------------------------------------------------------------------

timeframe_str = (
    f"{fetch_start_date.strftime('%Y-%m-%d')} "
    f"{fetch_end_date.strftime('%Y-%m-%d')}"
)


print(f"📅 Last recorded week : {last_recorded_date}")
print(f"📅 Current week start : {current_week_start}")
print(f"📅 Last completed week: {fetch_end_date}")
print(f"🚀 Fetching range      : {timeframe_str}\n")

📅 Last recorded week : 2026-08-28
📅 Current week start : 2026-09-27
📅 Last completed week: 2026-09-20
🚀 Fetching range      : 2026-08-28 2026-09-20



In [43]:
# ------------------------------------------------------------------
# 2. Configurations & Lookups
# ------------------------------------------------------------------
COUNTRY_MAP = {
    "US": "United_States",
    "IN": "India",
    "FR": "France",
    "NG": "Nigeria",
    "GB": "United_Kingdom",
    "CA": "Canada",
    "AU": "Australia",
    "BR": "Brazil",
    "MX": "Mexico",
    "ZA": "South_Africa",
    "KE": "Kenya",
    "SG": "Singapore",
    "AE": "United_Arab_Emirates",
    "CN": "China",
}

CATEGORY_THEME_MAP = {
    "Nutrition_Diets": [
        "HEALTH_DIET",
        "EATING",
        "FOOD_SECURITY",
        "AGRICULTURE",
    ],
    "Fitness_Wearables": ["HEALTH", "FITNESS", "TECHNOLOGY", "MED_DEVICE"],
    "Fashion_Beauty": ["FASHION", "BEAUTY", "RETAIL", "CONSUMER_GOODS"],
}

### Calculating features on GDELT data
- for Search_forecasting need 'Media_Volume_roll_mean_4w'
- for Market segmentation need 'Mean_Media_volume','Mean_Demand_to_hype_ratio','Mean_Net_Sentiment' 
- for Demand_to_Hype_Divergence_prediction need 'log_media_volume'

In [44]:
# ------------------------------------------------------------------
#  Set GCP Credentials & Client
# ------------------------------------------------------------------
import os
from datetime import datetime, date, timedelta
from google.cloud import bigquery
import pandas as pd

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = (
    r"C:\Users\Srivatsava\Downloads\gdelt-506803-be80fda77bde.json"
)

client = bigquery.Client(project="gdelt-506803")

# Target Date Parameters
target_start_date = last_recorded_date
target_end_date = fetch_end_date

# Parse target dates to datetime objects
if isinstance(target_start_date, str):
    start_dt = datetime.strptime(target_start_date, "%Y-%m-%d")
elif isinstance(target_start_date, date) and not isinstance(target_start_date, datetime):
    start_dt = datetime.combine(target_start_date, datetime.min.time())
else:
    start_dt = target_start_date

if isinstance(target_end_date, str):
    end_dt = datetime.strptime(target_end_date, "%Y-%m-%d")
elif isinstance(target_end_date, date) and not isinstance(target_end_date, datetime):
    end_dt = datetime.combine(target_end_date, datetime.min.time())
else:
    end_dt = target_end_date

# Extend query start date back by 28 days (4 weeks) to populate rolling lookbacks
lookback_start_dt = start_dt - timedelta(days=28)

start_str = lookback_start_dt.strftime("%Y-%m-%d")
end_str = end_dt.strftime("%Y-%m-%d")
target_start_str = start_dt.strftime("%Y-%m-%d")

print(
    f"🚀 Executing GDELT BigQuery extraction with 4-week lookback buffer: "
    f"{start_str} to {end_str} (Target Horizon: {target_start_str} to {end_str})"
)

# ------------------------------------------------------------------
# 1. BigQuery SQL Query
# ------------------------------------------------------------------
query = """
SELECT
  DATE_TRUNC(
    PARSE_DATE('%Y%m%d', SUBSTR(CAST(DATE AS STRING), 1, 8)),
    WEEK(SUNDAY)
  ) AS Week_Start,

  CASE
    WHEN V2Locations LIKE '%#US#%' THEN 'US'
    WHEN V2Locations LIKE '%#IN#%' THEN 'IN'
    WHEN V2Locations LIKE '%#DE#%' THEN 'DE'
    WHEN V2Locations LIKE '%#FR#%' THEN 'FR'
    WHEN V2Locations LIKE '%#NG#%' THEN 'NG'
    WHEN V2Locations LIKE '%#GB#%' THEN 'GB'
    WHEN V2Locations LIKE '%#CA#%' THEN 'CA'
    WHEN V2Locations LIKE '%#AU#%' THEN 'AU'
    WHEN V2Locations LIKE '%#BR#%' THEN 'BR'
    WHEN V2Locations LIKE '%#MX#%' THEN 'MX'
    WHEN V2Locations LIKE '%#ZA#%' THEN 'ZA'
    WHEN V2Locations LIKE '%#KE#%' THEN 'KE'
    WHEN V2Locations LIKE '%#SG#%' THEN 'SG'
    WHEN V2Locations LIKE '%#AE#%' THEN 'AE'
    WHEN V2Locations LIKE '%#JP#%' THEN 'JP'
    WHEN V2Locations LIKE '%#CH#%' THEN 'CN'
  END AS Country_Code,

  CASE
    WHEN V2Locations LIKE '%#US#%' THEN 'United_States'
    WHEN V2Locations LIKE '%#IN#%' THEN 'India'
    WHEN V2Locations LIKE '%#FR#%' THEN 'France'
    WHEN V2Locations LIKE '%#NG#%' THEN 'Nigeria'
    WHEN V2Locations LIKE '%#GB#%' THEN 'United_Kingdom'
    WHEN V2Locations LIKE '%#CA#%' THEN 'Canada'
    WHEN V2Locations LIKE '%#AU#%' THEN 'Australia'
    WHEN V2Locations LIKE '%#BR#%' THEN 'Brazil'
    WHEN V2Locations LIKE '%#MX#%' THEN 'Mexico'
    WHEN V2Locations LIKE '%#ZA#%' THEN 'South_Africa'
    WHEN V2Locations LIKE '%#KE#%' THEN 'Kenya'
    WHEN V2Locations LIKE '%#SG#%' THEN 'Singapore'
    WHEN V2Locations LIKE '%#AE#%' THEN 'United_Arab_Emirates'
    WHEN V2Locations LIKE '%#CH#%' THEN 'China'
  END AS Country_Name,

  CASE
    WHEN REGEXP_CONTAINS(LOWER(Themes), r'diet|nutrition|vegan|vegetarian|plant-based|keto|paleo|low-carb|low-fat|gluten-free|sugar-free|fasting|detox|organic|supplement|protein|vitamin|weight loss') THEN 'Nutrition_Diets'
    WHEN REGEXP_CONTAINS(LOWER(Themes), r'fitness|exercise|workout|training|gym|yoga|pilates|crossfit|running|wearable|smartwatch|tracker|garmin|fitbit') THEN 'Fitness_Wearables'
    WHEN REGEXP_CONTAINS(LOWER(Themes), r'fashion|clothing|apparel|style|designer|luxury|streetwear|athleisure|cosmetic|skincare|makeup|fragrance|spa') THEN 'Fashion_Beauty'
    ELSE 'Other'
  END AS Category,

  COUNT(*) AS Media_Volume,
  ROUND(AVG(SAFE_CAST(SPLIT(V2Tone, ',')[SAFE_OFFSET(0)] AS FLOAT64)), 2) AS tone_net_sentiment,
  ROUND(AVG(SAFE_CAST(SPLIT(V2Tone, ',')[SAFE_OFFSET(1)] AS FLOAT64)), 2) AS tone_positive_score,
  ROUND(AVG(SAFE_CAST(SPLIT(V2Tone, ',')[SAFE_OFFSET(2)] AS FLOAT64)), 2) AS tone_negative_score,
  ROUND(AVG(SAFE_CAST(SPLIT(V2Tone, ',')[SAFE_OFFSET(3)] AS FLOAT64)), 2) AS tone_polarity,
  ROUND(AVG(SAFE_CAST(SPLIT(V2Tone, ',')[SAFE_OFFSET(4)] AS FLOAT64)), 2) AS tone_activity_density,
  ROUND(AVG(SAFE_CAST(SPLIT(V2Tone, ',')[SAFE_OFFSET(5)] AS FLOAT64)), 2) AS tone_self_group_density,
  COUNT(DISTINCT SourceCommonName) AS Source_Diversity

FROM `gdelt-bq.gdeltv2.gkg_partitioned`
WHERE
  _PARTITIONTIME >= TIMESTAMP(@start_date)
  AND _PARTITIONTIME <= TIMESTAMP(@end_date)
  AND REGEXP_CONTAINS(LOWER(Themes), r'diet|nutrition|fitness|fashion')
  AND (
    V2Locations LIKE '%#US#%' OR V2Locations LIKE '%#IN#%' OR V2Locations LIKE '%#DE#%' OR V2Locations LIKE '%#FR#%' OR
    V2Locations LIKE '%#NG#%' OR V2Locations LIKE '%#GB#%' OR V2Locations LIKE '%#CA#%' OR V2Locations LIKE '%#AU#%' OR
    V2Locations LIKE '%#BR#%' OR V2Locations LIKE '%#MX#%' OR V2Locations LIKE '%#ZA#%' OR V2Locations LIKE '%#KE#%' OR
    V2Locations LIKE '%#SG#%' OR V2Locations LIKE '%#AE#%' OR V2Locations LIKE '%#JP#%' OR V2Locations LIKE '%#CH#%'
  )
  AND V2Tone IS NOT NULL
  AND LENGTH(V2Tone) > 5
GROUP BY Week_Start, Country_Code, Country_Name, Category
HAVING Week_Start IS NOT NULL AND Country_Code IS NOT NULL AND Category != 'Other'
ORDER BY Country_Name, Category, Week_Start;
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("start_date", "STRING", start_str),
        bigquery.ScalarQueryParameter("end_date", "STRING", end_str),
    ]
)

# Execute query and stream to Pandas
query_job = client.query(query, job_config=job_config)
df_gdelt = query_job.to_dataframe()

# Normalize column names to lowercase for consistency
df_gdelt.columns = df_gdelt.columns.str.lower()

# Enforce datetime type for week_start
df_gdelt["week_start"] = pd.to_datetime(df_gdelt["week_start"])

# Sort values
df_gdelt = df_gdelt.sort_values(
    by=["country_name", "category", "week_start"]
).reset_index(drop=True)

print(f"✅ Downloaded {len(df_gdelt)} rows (including lookback window).")

# Save to CSV (this will now have lowercase column names)
df_gdelt.to_csv("gdelt_data.csv", index=False)

🚀 Executing GDELT BigQuery extraction with 4-week lookback buffer: 2026-07-31 to 2026-09-20 (Target Horizon: 2026-08-28 to 2026-09-20)


c:\Users\Srivatsava\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✅ Downloaded 173 rows (including lookback window).


In [45]:
# ------------------------------------------------------------------
# 2. Clean & Merge Datasets (Eliminates Duplicate _x / _y Columns)
# ------------------------------------------------------------------
df_gdelt = pd.read_csv("gdelt_data.csv")
if "df_final_incremental" in locals() or "df_final_incremental" in globals():
    # Enforce datetime types
    df_final_incremental["week_start"] = pd.to_datetime(df_final_incremental["week_start"])
    df_gdelt["week_start"] = pd.to_datetime(df_gdelt["week_start"])

    # Clean df_final_incremental to keep ONLY unique, necessary trends columns
    # Dropping existing GDELT columns if they accidentally exist in df_final_incremental
    gdelt_cols_to_drop = [col for col in df_gdelt.columns if col not in ["week_start", "country_name", "category"] and col in df_final_incremental.columns]
    df_trends_clean = df_final_incremental.drop(columns=gdelt_cols_to_drop)

    # Clean up any trailing _x or _y suffixes if present in df_trends_clean
    df_trends_clean = df_trends_clean.loc[:, ~df_trends_clean.columns.str.endswith(('_x', '_y'))]

    # Perform a clean outer/left merge based on your primary keys
    df_merged = pd.merge(
        df_trends_clean,
        df_gdelt,
        on=["week_start", "country_name", "category"],
        how="left"
    )
else:
    df_merged = df_gdelt.copy()

# Sort data properly before running window calculations
df_merged = df_merged.sort_values(by=["country_name", "category", "week_start"]).reset_index(drop=True)

# ------------------------------------------------------------------
# 3. Handle Missing GDELT Media Volumes & Sentiment
# ------------------------------------------------------------------
# Missing Media_Volume or Source_Diversity means 0 media articles were found for that week
media_cols_zero_fill = ["media_volume", "source_diversity"]
for col in media_cols_zero_fill:
    if col in df_merged.columns:
        df_merged[col] = df_merged[col].fillna(0)

# Missing sentiment metrics fill with ffill() and bfill()
sentiment_cols = [
    "tone_net_sentiment", "tone_positive_score", "tone_negative_score",
    "tone_polarity", "tone_activity_density", "tone_self_group_density"
]

# Sort for chronological order
df_merged = df_merged.sort_values(["country_code", "category", "week_start"])

for col in sentiment_cols:
    if col in df_merged.columns:
        df_merged[col] = (
            df_merged.groupby(["country_code", "category"])[col]
            .transform(lambda x: x.ffill().bfill())
        )


# ------------------------------------------------------------------
#  4. Feature Engineering columns
# ------------------------------------------------------------------
grouped = df_merged.groupby(["country_name", "category"])

# 4-Week Rolling Mean Media Volume (fills early lookbacks with min_periods=1)
df_merged["media_volume_roll_mean_4w"] = grouped["media_volume"].transform(
    lambda x: x.shift(1).rolling(window=4, min_periods=1).mean()
).fillna(0)

# Log-transform media_volume safely (log1p handles zeros)
df_merged["log_media_volume"] = grouped["media_volume"].transform(lambda x: np.log1p(x))

# ------------------------------------------------------------------
# 5. Trim Buffer Data & Export Target Dataset
# ------------------------------------------------------------------
# Filter to target execution window
df_final = df_merged[
    df_merged["week_start"] >= pd.to_datetime(target_start_str)
].copy()

# Format Week_Start back to string YYYY-MM-DD
df_final["week_Start"] = df_final["week_start"].dt.strftime("%Y-%m-%d")

print("\n" + "=" * 70)
print("✅ GDELT & TRENDS MERGE COMPLETE (NO NULL VALUES)")
print("=" * 70)
print(f"Target Row Count   : {len(df_final):,}")
print(f"Null Values Count  :\n{df_final.isnull().sum()}")
print("=" * 70)
print(df_final.head())


✅ GDELT & TRENDS MERGE COMPLETE (NO NULL VALUES)
Target Row Count   : 168
Null Values Count  :
week_start                       0
country_name                     0
category                         0
search_interest                  0
search_velocity                  0
search_acceleration              0
search_interest_lag_1            0
search_interest_lag_2            0
search_interest_lag_4            0
search_velocity_lag              0
search_velocity_lag1             0
search_velocity_lag2             0
search_acceleration_lag          0
search_interest_roll_mean_4w     0
search_interest_roll_std_4w      0
country_code                    94
media_volume                     0
tone_net_sentiment              94
tone_positive_score             94
tone_negative_score             94
tone_polarity                   94
tone_activity_density           94
tone_self_group_density         94
source_diversity                 0
media_volume_roll_mean_4w        0
log_media_volume             

In [46]:
# removing country code columns form df_final
df_final = df_final.drop(columns='country_code')

In [47]:
# Get all columns with missing values
null_cols = df_final.columns[df_final.isnull().sum() > 0]
print(null_cols)

Index(['tone_net_sentiment', 'tone_positive_score', 'tone_negative_score',
       'tone_polarity', 'tone_activity_density', 'tone_self_group_density'],
      dtype='object')


In [48]:
for col in null_cols:
    df_final[col] = df_final[col].ffill()

In [49]:
df_final.isnull().sum()

week_start                      0
country_name                    0
category                        0
search_interest                 0
search_velocity                 0
search_acceleration             0
search_interest_lag_1           0
search_interest_lag_2           0
search_interest_lag_4           0
search_velocity_lag             0
search_velocity_lag1            0
search_velocity_lag2            0
search_acceleration_lag         0
search_interest_roll_mean_4w    0
search_interest_roll_std_4w     0
media_volume                    0
tone_net_sentiment              0
tone_positive_score             0
tone_negative_score             0
tone_polarity                   0
tone_activity_density           0
tone_self_group_density         0
source_diversity                0
media_volume_roll_mean_4w       0
log_media_volume                0
week_Start                      0
dtype: int64

In [50]:
path = r'E:\\ML Project Internship\\ML-Project\\data\\processed\\'
df_final.to_csv(path + 'user_input_Merged_data.csv',index= False)

### Holidays and Macro data

In [51]:
# Target Date Parameters
target_start_date = last_recorded_date
target_end_date = fetch_end_date

In [52]:
from datetime import timedelta
import holidays
import pandas as pd

# ------------------------------------------------------------------
# 1. Country Name to ISO Alpha-2 Mapping for Python `holidays`
# ------------------------------------------------------------------
COUNTRY_MAP = {
    "United_States": "US",
    "India": "IN",
    "France": "FR",
    "Nigeria": "NG",
    "United_Kingdom": "GB",
    "Canada": "CA",
    "Australia": "AU",
    "Brazil": "BR",
    "Mexico": "MX",
    "South_Africa": "ZA",
    "Kenya": "KE",
    "Singapore": "SG",
    "United_Arab_Emirates": "AE",
    "China": "CN",
    "Germany": "DE",
    "Japan": "JP",
}

# ------------------------------------------------------------------
# 2. Holiday Feature Calculation Function
# ------------------------------------------------------------------
# Pre-cache country holiday instances to avoid re-instantiating inside loops
years_to_cover = df_final["week_start"].astype(str).str[:4].unique().astype(int)
holiday_dict = {}

for country_name, iso_code in COUNTRY_MAP.items():
    try:
        holiday_dict[country_name] = holidays.country_holidays(
            iso_code, years=years_to_cover
        )
    except NotImplementedError:
        # Fallback if a specific country code isn't supported in holidays library
        holiday_dict[country_name] = {}


def get_weekly_holidays(row):
    country = row["country_name"]
    week_start = row["week_start_dt"]

    # Retrieve country holiday calendar
    country_holidays = holiday_dict.get(country, {})
    if not country_holidays:
        return 0

    # Count holidays within the 7-day window [Week_Start, Week_Start + 6 days]
    week_days = [week_start + timedelta(days=i) for i in range(7)]
    count = sum(1 for day in week_days if day in country_holidays)

    return count


# ------------------------------------------------------------------
# 3. Apply Feature Engineering to DataFrame
# ------------------------------------------------------------------
# Temporarily ensure Week_Start is datetime format
df_final["week_start_dt"] = pd.to_datetime(df_final["week_start"])

# Calculate holiday_count across the 7-day week
df_final["holiday_count"] = df_final.apply(get_weekly_holidays, axis=1)

# Calculate binary indicator is_holiday
df_final["is_holiday"] = (df_final["holiday_count"] > 0).astype(int)

# Clean up helper column
df_final = df_final.drop(columns=["week_start_dt"])

print("\n" + "=" * 70)
print("✅ HOLIDAY FEATURES SUCCESSFULLY CREATED (NO NULL VALUES)")
print("=" * 70)
print(
    f"Holiday Feature Nulls:\n{df_final[['holiday_count', 'is_holiday']].isnull().sum()}"
)
print("=" * 70)
print(
    df_final[
        ["week_start", "country_name", "category", "holiday_count", "is_holiday"]
    ].head(10)
)


✅ HOLIDAY FEATURES SUCCESSFULLY CREATED (NO NULL VALUES)
Holiday Feature Nulls:
holiday_count    0
is_holiday       0
dtype: int64
    week_start          country_name           category  holiday_count  \
140 2026-08-30  United_Arab_Emirates    Nutrition_Diets              0   
141 2026-09-06  United_Arab_Emirates    Nutrition_Diets              0   
142 2026-09-13  United_Arab_Emirates    Nutrition_Diets              0   
143 2026-09-20  United_Arab_Emirates    Nutrition_Diets              0   
5   2026-09-06             Australia  Fitness_Wearables              0   
8   2026-08-30             Australia    Nutrition_Diets              0   
9   2026-09-06             Australia    Nutrition_Diets              0   
10  2026-09-13             Australia    Nutrition_Diets              0   
11  2026-09-20             Australia    Nutrition_Diets              0   
16  2026-08-30                Brazil  Fitness_Wearables              0   

     is_holiday  
140           0  
141           0  

In [53]:
path = r'E:\\ML Project Internship\\ML-Project\\data\\processed\\'
df_final.to_csv(path + 'user_input_Merged_data.csv',index= False)

In [54]:
df_final = pd.read_csv("E:\\ML Project Internship\\ML-Project\\data\\processed\\user_input_Merged_data.csv")

In [55]:
import pandas as pd
import requests

# ------------------------------------------------------------------
# 1. Country ISO Mapping & World Bank Indicator Codes
# ------------------------------------------------------------------
COUNTRY_ISO_MAP = {
    "United_States": "USA",
    "India": "IND",
    "France": "FRA",
    "Nigeria": "NGA",
    "United_Kingdom": "GBR",
    "Canada": "CAN",
    "Australia": "AUS",
    "Brazil": "BRA",
    "Mexico": "MEX",
    "South_Africa": "ZAF",
    "Kenya": "KEN",
    "Singapore": "SGP",
    "United_Arab_Emirates": "ARE",
    "China": "CHN",
    "Germany": "DEU",
    "Japan": "JPN",
}

INDICATORS = {
    "Inflation_Rate": "FP.CPI.TOTL.ZG",
    "Internet_Penetration": "IT.NET.USER.ZS",
    "GDP_Per_Capita": "NY.GDP.PCAP.CD",
}

# ------------------------------------------------------------------
# 2. Fetch World Bank Indicators
# ------------------------------------------------------------------
country_codes_str = ";".join(COUNTRY_ISO_MAP.values())
wb_records = []

print("🌐 Fetching latest available World Bank indicators...")

for col_name, indicator_code in INDICATORS.items():
    # Fetch historical window (2015 to current)
    url = (
        f"http://api.worldbank.org/v2/country/{country_codes_str}/indicator/{indicator_code}"
        f"?date=2015:2026&format=json&per_page=1000"
    )

    try:
        response = requests.get(url, timeout=15)
        if response.status_code == 200:
            data = response.json()
            if len(data) > 1 and data[1]:
                for entry in data[1]:
                    val = entry.get("value")
                    if val is not None:
                        wb_records.append(
                            {
                                "ISO3": entry["countryiso3code"],
                                "Year": int(entry["date"]),
                                "Metric": col_name,
                                "Value": float(val),
                            }
                        )
    except Exception as e:
        print(f"⚠️ Warning: Failed fetching {col_name}: {e}")

# ------------------------------------------------------------------
# 3. Extract Latest Valid Value Per Country
# ------------------------------------------------------------------
df_wb_raw = pd.DataFrame(wb_records)

# Reverse map ISO3 back to Country_Name
iso_to_country = {v: k for k, v in COUNTRY_ISO_MAP.items()}
df_wb_raw["country_name"] = df_wb_raw["ISO3"].map(iso_to_country)

# Sort by Year ascending so the last entry for each country is the most recent
df_wb_raw = df_wb_raw.sort_values(by="Year")

# Get the latest available value for each Country and Metric
df_latest_macro = (
    df_wb_raw.groupby(["country_name", "Metric"])["Value"]
    .last()
    .unstack()
    .reset_index()
)

# ------------------------------------------------------------------
# 4. Merge Latest Macro Metrics into Main DataFrame
# ------------------------------------------------------------------
macro_cols = ["inflation_rate", "internet_penetration", "gdp_per_capita"]

# Ensure existing columns are dropped prior to merging to prevent duplicates
df_final = df_final.drop(columns=[c for c in macro_cols if c in df_final.columns], errors="ignore")

# Merge strictly on Country_Name
df_final = pd.merge(df_final, df_latest_macro, on="country_name", how="left")

# Fallback: Fill any remaining unmapped country values with 0
for col in macro_cols:
    if col in df_final.columns:
        df_final[col] = df_final[col].fillna(0.0)
    else:
        df_final[col] = 0.0

print("\n" + "=" * 70)
print("✅ WORLD BANK MACROECONOMIC FEATURES SUCCESSFULLY ADDED")
print("=" * 70)
print(f"Null Values Count  :\n{df_final[macro_cols].isnull().sum()}")
print("=" * 70)
print(
    df_final[
        ["week_start", "country_name", "inflation_rate", "internet_penetration", "gdp_per_capita"]
    ].head(10)
)

🌐 Fetching latest available World Bank indicators...

✅ WORLD BANK MACROECONOMIC FEATURES SUCCESSFULLY ADDED
Null Values Count  :
inflation_rate          0
internet_penetration    0
gdp_per_capita          0
dtype: int64
   week_start          country_name  inflation_rate  internet_penetration  \
0  2026-08-30  United_Arab_Emirates             0.0                   0.0   
1  2026-09-06  United_Arab_Emirates             0.0                   0.0   
2  2026-09-13  United_Arab_Emirates             0.0                   0.0   
3  2026-09-20  United_Arab_Emirates             0.0                   0.0   
4  2026-09-06             Australia             0.0                   0.0   
5  2026-08-30             Australia             0.0                   0.0   
6  2026-09-06             Australia             0.0                   0.0   
7  2026-09-13             Australia             0.0                   0.0   
8  2026-09-20             Australia             0.0                   0.0   
9  2026-0

In [57]:
## Calculating Demand to Hype Ratio
df_final['demand_to_hype_ratio'] = (df_final['search_interest'] /(df_final['media_volume'] + 1))

In [58]:
path = r'E:\\ML Project Internship\\ML-Project\\data\\processed\\'
df_final.to_csv(path + 'user_input_Merged_data.csv',index= False)

### Creating seperate data files for each model so they can be used directly by fastapi code 

In [59]:
import pandas as pd
path = r'E:\\ML Project Internship\\ML-Project\\data\\processed\\'
df_final = pd.read_csv(path + 'user_input_Merged_data.csv')

In [60]:
columns = df_final.columns

In [61]:
columns

Index(['week_start', 'country_name', 'category', 'search_interest',
       'search_velocity', 'search_acceleration', 'search_interest_lag_1',
       'search_interest_lag_2', 'search_interest_lag_4', 'search_velocity_lag',
       'search_velocity_lag1', 'search_velocity_lag2',
       'search_acceleration_lag', 'search_interest_roll_mean_4w',
       'search_interest_roll_std_4w', 'media_volume', 'tone_net_sentiment',
       'tone_positive_score', 'tone_negative_score', 'tone_polarity',
       'tone_activity_density', 'tone_self_group_density', 'source_diversity',
       'media_volume_roll_mean_4w', 'log_media_volume', 'week_Start',
       'holiday_count', 'is_holiday', 'GDP_Per_Capita', 'Inflation_Rate',
       'Internet_Penetration', 'inflation_rate', 'internet_penetration',
       'gdp_per_capita', 'demand_to_hype_ratio'],
      dtype='object')

In [62]:
Search_Intrest_forecasting_features = ['week_start', 'country_name', 'category', 'search_interest',
       'demand_to_hype_ratio', 'inflation_rate', 'gdp_per_capita',
       'holiday_count', 'search_interest_lag_1', 'search_interest_lag_2',
       'search_interest_lag_4', 'search_velocity_lag',
       'search_acceleration_lag', 'search_interest_roll_mean_4w',
       'search_interest_roll_std_4w', 'media_volume_roll_mean_4w', 'sin_week',
       'cos_week']

In [63]:
set(Search_Intrest_forecasting_features)-set(columns) 

{'cos_week', 'sin_week'}

### Calculating pending features for Search_intrest forecasting

In [64]:
# Week_Start is datetime
df_final["week_start"] = pd.to_datetime(df_final["week_start"])

# Now you can safely use .dt
week_of_year = df_final["week_start"].dt.isocalendar().week
df_final['sin_week'] = np.sin(2 * np.pi * week_of_year / 52)
df_final['cos_week'] = np.cos(2 * np.pi * week_of_year / 52)

In [65]:
Search_Intrest_forecasting_features_df = df_final[Search_Intrest_forecasting_features].copy()

### Calculating pending features for Demand_to_hype_Divergence_predcition

In [66]:
Demand_ot_hype_Divergence_score_features = ['country_name', 'category', 'search_velocity', 'search_acceleration',
       'tone_net_sentiment', 'tone_polarity', 'tone_activity_density',
       'source_diversity', 'inflation_rate', 'internet_penetration',
       'holiday_count', 'log_media_volume', 'search_velocity_lag1',
       'search_velocity_lag2']

In [67]:
set(Demand_ot_hype_Divergence_score_features) - set(columns)

set()

### No need to caluclate fetures already exist

### Calculating pending features for Market_segmentation

In [68]:
Market_segmentation_features = ['Country_Name', 'Category', 'Mean_Search_Interest', 'Mean_Media_Volume',
       'Mean_Demand_to_Hype_Ratio', 'Search_Interest_Trend_Slope',
       'Mean_Net_Sentiment', 'Mean_GDP_Per_Capita']

In [69]:
mean_search = (df_final.groupby(["country_name", "category"])["search_interest"].mean().reset_index(name="mean_search_interest"))

In [70]:
# Create a new DataFrame for market segmentation
market_segmentation = mean_search.copy()

In [71]:
mean_media_volume = (df_final.groupby(["country_name", "category"])["media_volume"].mean().reset_index(name="mean_media_volume"))
market_segmentation = pd.merge(market_segmentation, mean_media_volume, on=["country_name", "category"])

In [72]:
## Calculating Demand to Hype Ratio
df_final['demand_to_hype_ratio'] = (df_final['search_interest'] /(df_final['media_volume'] + 1))

In [73]:
mean_demand_to_hype_ratio = (df_final.groupby(["country_name", "category"])["demand_to_hype_ratio"].mean().reset_index(name="mean_demand_to_hype_ratio"))
market_segmentation = pd.merge(market_segmentation, mean_demand_to_hype_ratio , on=["country_name", "category"])

In [74]:
def calculate_slope(group):
    group = group.sort_values("week_start")
    x = (group["week_start"] - group["week_start"].min()).dt.days // 7 + 1
    y = group["search_interest"]

    x_mean, y_mean = x.mean(), y.mean()
    denominator = np.sum((x - x_mean) ** 2)

    return (
        np.sum((x - x_mean) * (y - y_mean)) / denominator
        if denominator != 0
        else 0
    )

search_interest_trend_slope = (
    df_final.groupby(["country_name", "category"])
    .apply(calculate_slope)
    .reset_index(name="search_interest_trend_slope")
)
market_segmentation = pd.merge(market_segmentation, search_interest_trend_slope, on=["country_name", "category"])

C:\Users\Srivatsava\AppData\Local\Temp\ipykernel_29660\3941833743.py:17: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calculate_slope)


In [75]:
mean_net_sentiment = (
    df_final.groupby(["country_name", "category"])["tone_net_sentiment"]
    .mean()
    .reset_index(name="mean_net_sentiment")
)
market_segmentation = pd.merge(market_segmentation, mean_net_sentiment, on=["country_name", "category"])

In [76]:
mean_gdp_per_capita = (
    df_final.groupby(["country_name", "category"])["gdp_per_capita"]
    .mean()
    .reset_index(name="mean_gdp_per_capita")
)
market_segmentation = pd.merge(market_segmentation, mean_gdp_per_capita, on=["country_name", "category"])

In [77]:
import io
import os
import pandas as pd
import boto3
from botocore.config import Config
from dotenv import load_dotenv

# ------------------------------------------------------------------
# 1. Load Environment Variables & Configure R2 Client
# ------------------------------------------------------------------
load_dotenv(override=True)

R2_ACCOUNT_ID = os.getenv("R2_ACCOUNT_ID", "").strip()
R2_ACCESS_KEY_ID = os.getenv("R2_ACCESS_KEY_ID", "").strip()
R2_SECRET_ACCESS_KEY = os.getenv("R2_SECRET_ACCESS_KEY", "").strip()
R2_BUCKET_NAME = os.getenv("R2_BUCKET_NAME", "").strip().strip("/")

if not all([R2_ACCOUNT_ID, R2_ACCESS_KEY_ID, R2_SECRET_ACCESS_KEY, R2_BUCKET_NAME]):
    raise ValueError("❌ Missing R2 environment variables in .env file.")

endpoint_url = f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com"

s3_client = boto3.client(
    "s3",
    endpoint_url=endpoint_url,
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_ACCESS_KEY,
    region_name="us-east-1",
    config=Config(signature_version="s3v4", s3={"addressing_style": "path"})
)

def upload_df_csv_to_r2(df: pd.DataFrame, r2_key: str) -> bool:
    """Converts a pandas DataFrame to CSV bytes and uploads directly to R2."""
    clean_key = r2_key.lstrip("/")
    try:
        print(f"⏳ Encoding DataFrame to CSV for '{clean_key}'...", end="", flush=True)
        csv_buffer = io.StringIO()
        df.to_csv(csv_buffer, index=False)
        payload_bytes = csv_buffer.getvalue().encode("utf-8")
        size_kb = len(payload_bytes) / 1024
        print(f" ✅ Done ({size_kb:.2f} KB)")

        print(f"⏳ Uploading to R2 bucket '{R2_BUCKET_NAME}'...", end="", flush=True)
        response = s3_client.put_object(
            Bucket=R2_BUCKET_NAME,
            Key=clean_key,
            Body=payload_bytes,
            ContentType="text/csv"
        )
        
        status_code = response.get("ResponseMetadata", {}).get("HTTPStatusCode")
        if status_code == 200:
            print(f" ✅ Success! (HTTP 200) -> {R2_BUCKET_NAME}/{clean_key}")
            return True
        else:
            print(f" ⚠️ Unexpected HTTP status: {status_code}")
            return False
    except Exception as e:
        print(f"\n❌ Upload failed for '{clean_key}': {e}")
        return False

# ------------------------------------------------------------------
# 2. Direct R2 Upload Pipeline (Batch CSV Datasets)
# ------------------------------------------------------------------
# Prepare dataframes
Demand_to_hype_Divergence_score_features_df = df_final[Demand_ot_hype_Divergence_score_features].copy()

csv_datasets = [
    ("Demand_to_hype_Divergence_score_features_df",
     Demand_to_hype_Divergence_score_features_df,
     "Demand_to_hype_Divergence_score_features.csv"),
    ("market_segmentation",
     market_segmentation,
     "market_segmentation.csv"),
    ("Search_Intrest_forecasting_features_df",
     Search_Intrest_forecasting_features_df,
     "Search_Intrest_forecasting_features.csv")
]

print("🚀 Starting batch upload process to Cloudflare R2...\n")

for var_name, df, file_name in csv_datasets:
    r2_key = f"data/processed/{file_name}"
    upload_df_csv_to_r2(df, r2_key)
    print("-" * 65)

print("\n🎉 All CSV datasets successfully uploaded to Cloudflare R2!")


🚀 Starting batch upload process to Cloudflare R2...

⏳ Encoding DataFrame to CSV for 'data/processed/Demand_to_hype_Divergence_score_features.csv'... ✅ Done (22.86 KB)
⏳ Uploading to R2 bucket 'marketing-intelligence'... ✅ Success! (HTTP 200) -> marketing-intelligence/data/processed/Demand_to_hype_Divergence_score_features.csv
-----------------------------------------------------------------
⏳ Encoding DataFrame to CSV for 'data/processed/market_segmentation.csv'... ✅ Done (3.70 KB)
⏳ Uploading to R2 bucket 'marketing-intelligence'... ✅ Success! (HTTP 200) -> marketing-intelligence/data/processed/market_segmentation.csv
-----------------------------------------------------------------
⏳ Encoding DataFrame to CSV for 'data/processed/Search_Intrest_forecasting_features.csv'... ✅ Done (31.57 KB)
⏳ Uploading to R2 bucket 'marketing-intelligence'... ✅ Success! (HTTP 200) -> marketing-intelligence/data/processed/Search_Intrest_forecasting_features.csv
----------------------------------------

In [78]:
df_final.columns

Index(['week_start', 'country_name', 'category', 'search_interest',
       'search_velocity', 'search_acceleration', 'search_interest_lag_1',
       'search_interest_lag_2', 'search_interest_lag_4', 'search_velocity_lag',
       'search_velocity_lag1', 'search_velocity_lag2',
       'search_acceleration_lag', 'search_interest_roll_mean_4w',
       'search_interest_roll_std_4w', 'media_volume', 'tone_net_sentiment',
       'tone_positive_score', 'tone_negative_score', 'tone_polarity',
       'tone_activity_density', 'tone_self_group_density', 'source_diversity',
       'media_volume_roll_mean_4w', 'log_media_volume', 'week_Start',
       'holiday_count', 'is_holiday', 'GDP_Per_Capita', 'Inflation_Rate',
       'Internet_Penetration', 'inflation_rate', 'internet_penetration',
       'gdp_per_capita', 'demand_to_hype_ratio', 'sin_week', 'cos_week'],
      dtype='object')

In [79]:
col = ['week_start',
 'country_name',
 'category',
 'search_interest',
 'search_velocity',
 'search_acceleration',
 'media_volume',
 'tone_net_sentiment',
 'tone_positive_score',
 'tone_negative_score',
 'tone_polarity',
 'tone_activity_density',
 'tone_self_group_density',
 'source_diversity',
 'demand_to_hype_ratio',
 'inflation_rate',
 'internet_penetration',
 'gdp_per_capita',
 'is_holiday',
 'holiday_count']

In [80]:
set(df_final)-set(col)

{'GDP_Per_Capita',
 'Inflation_Rate',
 'Internet_Penetration',
 'cos_week',
 'log_media_volume',
 'media_volume_roll_mean_4w',
 'search_acceleration_lag',
 'search_interest_lag_1',
 'search_interest_lag_2',
 'search_interest_lag_4',
 'search_interest_roll_mean_4w',
 'search_interest_roll_std_4w',
 'search_velocity_lag',
 'search_velocity_lag1',
 'search_velocity_lag2',
 'sin_week',
 'week_Start'}

In [81]:
set(col)-set(df_final)

set()

In [82]:
final_incremental_data = df_final[col].copy()

In [83]:
csv_dataset = [
    ("final_incremental_data",
     final_incremental_data,
     "final_incremental_data.csv"),
]

print("🚀 Started uploading final_incremental_data to Cloudflare R2...\n")

for var_name, df, file_name in csv_dataset:
    r2_key = f"data/processed/{file_name}"
    upload_df_csv_to_r2(df, r2_key)
    print("-" * 65)

print("\n🎉 final_incremental_data CSV datasets is successfully uploaded to Cloudflare R2!")

🚀 Started uploading final_incremental_data to Cloudflare R2...

⏳ Encoding DataFrame to CSV for 'data/processed/final_incremental_data.csv'... ✅ Done (24.26 KB)
⏳ Uploading to R2 bucket 'marketing-intelligence'... ✅ Success! (HTTP 200) -> marketing-intelligence/data/processed/final_incremental_data.csv
-----------------------------------------------------------------

🎉 final_incremental_data CSV datasets is successfully uploaded to Cloudflare R2!


### Updating a week of data to main table in TIDB

In [86]:
import os
import sys
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# 1. Load Environment Variables
load_dotenv()

DB_HOST = os.getenv("DB_HOST")
DB_PORT = int(os.getenv("DB_PORT", 4000))
DB_USERNAME = os.getenv("DB_USERNAME")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_DATABASE = os.getenv("DB_DATABASE")

# 2. Create Engine
connection_url = (
    f"mysql+pymysql://{DB_USERNAME}:{DB_PASSWORD}@"
    f"{DB_HOST}:{DB_PORT}/{DB_DATABASE}"
)

engine = create_engine(
    connection_url,
    connect_args={"ssl": {}} if "tidbcloud" in DB_HOST else {}
)

# 3. Query Max Date Entry
max_date_query = text("""
    SELECT MAX(STR_TO_DATE(week_start, '%Y-%m-%d')) AS latest_week
    FROM main;
""")

with engine.connect() as connection:
    latest_db_date = connection.execute(max_date_query).scalar()
    
    # Format to string 'YYYY-MM-DD'
    latest_db_date_str = latest_db_date.strftime("%Y-%m-%d") if latest_db_date else None
    
    print(f"Latest week_start in 'main': {latest_db_date_str}")

Latest week_start in 'main': 2026-08-23


In [87]:
import io
import boto3
import pandas as pd

# ------------------------------------------------------------------
# 1. Load Credentials & Fetch File from R2
# ------------------------------------------------------------------
R2_ACCOUNT_ID = os.getenv("R2_ACCOUNT_ID")
R2_ACCESS_KEY_ID = os.getenv("R2_ACCESS_KEY_ID")
R2_SECRET_ACCESS_KEY = os.getenv("R2_SECRET_ACCESS_KEY")
R2_BUCKET_NAME = os.getenv("R2_BUCKET_NAME")

r2_key = "data/processed/final_incremental_data.csv"

r2_client = boto3.client(
    "s3",
    endpoint_url=f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_ACCESS_KEY,
    region_name="auto"
)

print(f"📥 Fetching '{r2_key}' from Cloudflare R2...")

response = r2_client.get_object(Bucket=R2_BUCKET_NAME, Key=r2_key)
csv_bytes = response["Body"].read()
df_r2 = pd.read_csv(io.BytesIO(csv_bytes))

# Standardize date format to string 'YYYY-MM-DD'
df_r2["week_start"] = pd.to_datetime(df_r2["week_start"]).dt.strftime("%Y-%m-%d")

# ------------------------------------------------------------------
# 2. Dynamically Find the Immediate Next Target Week & Filter
# ------------------------------------------------------------------
# Filter for dates strictly greater than the DB's latest date (2026-08-23)
future_weeks = df_r2[df_r2["week_start"] > latest_db_date_str]["week_start"].unique()

if len(future_weeks) == 0:
    print(f"\nℹ️ No new weeks found in R2 CSV beyond {latest_db_date_str}.")
    df_next_week = pd.DataFrame()
else:
    # Get the earliest date among all future weeks (immediate next week)
    target_date = min(future_weeks)
    
    # Filter DataFrame strictly for the target week
    df_next_week = df_r2[df_r2["week_start"] == target_date].copy()

    print(f"\n✓ DB Latest Date: {latest_db_date_str}")
    print(f"🎯 Calculated Target Week: {target_date}")
    print(f"📊 Prepared {len(df_next_week)} rows for insertion.")

📥 Fetching 'data/processed/final_incremental_data.csv' from Cloudflare R2...

✓ DB Latest Date: 2026-08-23
🎯 Calculated Target Week: 2026-08-30
📊 Prepared 42 rows for insertion.


In [88]:
# ------------------------------------------------------------------
# Insert Next Week Data into TiDB 'main' Table
# ------------------------------------------------------------------
if df_next_week.empty:
    print("ℹ️ No target week data available to insert.")
else:
    target_date = df_next_week["week_start"].iloc[0]
    records_count = len(df_next_week)

    print(f"🚀 Appending {records_count} rows for week '{target_date}' to 'main' table...")

    try:
        # Append data to the existing 'main' table
        df_next_week.to_sql(
            name="main",
            con=engine,
            if_exists="append",
            index=False,
            chunksize=1000
        )
        print(f"✅ Successfully appended {records_count} rows for week {target_date}!")

        # Double check new max date in TiDB
        with engine.connect() as connection:
            new_max = connection.execute(
                text("SELECT MAX(STR_TO_DATE(week_start, '%Y-%m-%d')) FROM main;")
            ).scalar()
            new_max_str = new_max.strftime("%Y-%m-%d") if new_max else "N/A"

        print(f"📌 Updated Latest Week in TiDB: {new_max_str}")

    except Exception as e:
        print(f"✗ Failed to append data: {e}")

🚀 Appending 42 rows for week '2026-08-30' to 'main' table...
✅ Successfully appended 42 rows for week 2026-08-30!
📌 Updated Latest Week in TiDB: 2026-08-30
